In [ ]:
!pip install --upgrade --quiet google-genai

In [ ]:
from io import BytesIO
from PIL import Image
from google.cloud import storage
from google import genai
from google.genai import types

storage_client = storage.Client()

PROJECT_ID = "qwiklabs-gcp-04-22924826f084"
BUCKET_NAME = f"{PROJECT_ID}-bucket"
NANO_BANANA = "gemini-2.5-flash-image"

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location="global",
)

In [ ]:
def display_image(image_data):
    """Displays an image in a Colab notebook from PNG data."""
    img = Image.open(BytesIO(image_data))
    img.save('my_image.png')
    display(img)

def display_img_from_gcs(bucket_name, img_path):
    """Downloads bytes and displays an image stored in GCS."""
    bucket = storage_client.get_bucket(bucket_name)
    blob = bucket.get_blob(img_path)
    display_image(blob.download_as_bytes())

In [ ]:
image1 = types.Part.from_uri(
    file_uri=f"gs://{BUCKET_NAME}/product_dress_128.png",
    mime_type="image/png",
)

image2 = types.Part.from_uri(
    file_uri=f"gs://{BUCKET_NAME}/product_jacket_932.png",
    mime_type="image/png",
)

prompt = types.Part.from_text(
    text=("Create a photo featuring a young couple"
          "from Austin, TX wearing these clothing"
          "products on a walk through the Greenbelt.")
)

content = [
    types.Content(
        role="user",
        parts=[image1, image2, prompt]
    )
]

In [ ]:
gen_config = types.GenerateContentConfig(
    response_modalities = ["TEXT", "IMAGE"],
    http_options = types.HttpOptions(
        retry_options = types.HttpRetryOptions(
            attempts = 60,
            initial_delay = 1,
            exp_base = 2,
            max_delay = 4
        )
    )
)

In [ ]:
response = client.models.generate_content(
    model = NANO_BANANA,
    contents = content,
    config = gen_config,
    )

for part in response.candidates[0].content.parts:
    if part.inline_data:
        image_data = part.inline_data.data
        display_image(image_data)

## Edit a UI with Nano Banana

In [ ]:
display_img_from_gcs(BUCKET_NAME, "lyria_ui.png")

In [ ]:
ui_img_part = types.Part.from_uri(
    file_uri=f"gs://{BUCKET_NAME}/lyria_ui.png",
    mime_type="image/png",
)
text1 = types.Part.from_text(
    text="""
    Create a UI in the same style.
    It should be called Podcast Studio.
    Include configuration options:
    - A UI slider to set the number of hosts
      (from 1 to max 4)
    - A UI slider to set the minutes in length
      (from 2 to max 40)
    The prompt input box should tell the user to
    specify details of the episode topic.
    """
)

contents = [
    types.Content(
        role="user",
        parts=[ui_img_part, text1]
    )
]

response = client.models.generate_content(
    model = NANO_BANANA,
    contents = contents,
    config = gen_config,
    )

for part in response.candidates[0].content.parts:
    if part.inline_data:
        image_data = part.inline_data.data
        display_image(image_data)

In [ ]:
bucket = storage_client.get_bucket(BUCKET_NAME)
blob = bucket.blob("original_ui.png")
blob.upload_from_filename("my_image.png")

original_ui_img_part = types.Part.from_uri(
    file_uri=f"gs://{BUCKET_NAME}/original_ui.png",
    mime_type="image/png",
)
text = types.Part.from_text(
    text="""
        Modify this image to remove the
        "Advanced Options" area. Keep current
        configuration options (Number of hosts
        and Minutes in Length) and add a new
        dropdown for Podcast Language.
    """
)

contents = [
    types.Content(
        role="user",
        parts=[original_ui_img_part, text]
    )
]

response = client.models.generate_content(
    model = NANO_BANANA,
    contents = contents,
    config = gen_config,
    )

for part in response.candidates[0].content.parts:
    if part.inline_data:
        image_data = part.inline_data.data
        display_image(image_data)